<a href="https://colab.research.google.com/github/Navya40869/edge-idps-colab/blob/main/models/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("Is GPU available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

Is GPU available?: True
GPU Device Name: Tesla T4


In [1]:
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# Load preprocessed arrays directly from Drive
save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"
X_train = np.load(save_path + "X_train.npy")
y_train = np.load(save_path + "y_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_test = np.load(save_path + "y_test.npy")

print(f"Loaded X_train shape: {X_train.shape}")

Mounted at /content/drive
Loaded X_train shape: (71538, 39)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Check GPU connectivity
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create PyTorch DataLoaders for mini-batch processing
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")

Using device: cuda
Number of training batches: 559


In [5]:
from sklearn.preprocessing import LabelEncoder

# Encode labels to be contiguous integers starting from 0 (e.g., 0, 1, 2, ..., N-1)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# Verify updated class parameters
num_classes = len(label_encoder.classes_)
print(f"Number of target classes: {num_classes}")
print(f"Unique encoded labels in y_train: {np.unique(y_train)}")

Number of target classes: 29
Unique encoded labels in y_train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28]


In [1]:
import os
import joblib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# ==========================================
# 1. LOAD DATA & RE-ENCODE LABELS
# ==========================================
save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"

# Load raw numpy arrays from Google Drive
X_train = np.load(save_path + "X_train.npy")
y_train = np.load(save_path + "y_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_test = np.load(save_path + "y_test.npy")

# Encode labels to consecutive integers (0 to num_classes - 1)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

num_features = X_train.shape[1]
num_classes = len(label_encoder.classes_)

print(f"Dataset loaded successfully!")
print(f"Features: {num_features} | Classes: {num_classes}")
print(f"Encoded class range: {y_train.min()} to {y_train.max()}")

# ==========================================
# 2. CONVERT TO PYTORCH TENSORS & DATALOADERS
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================================
# 3. DEFINE MODEL ARCHITECTURE
# ==========================================
class EdgeIDPSClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EdgeIDPSClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = EdgeIDPSClassifier(num_features, num_classes).to(device)

# ==========================================
# 4. RUN TRAINING LOOP
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 15

print("\nStarting training on T4 GPU...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

# ==========================================
# 5. EVALUATE AND SAVE ARTIFACTS
# ==========================================
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

print("\n--- Model Evaluation Report ---")
print(classification_report(all_targets, all_preds))

# Save trained PyTorch model and updated label encoder for Member 3 & 4
torch.save(model.state_dict(), save_path + "edge_idps_model.pth")
joblib.dump(label_encoder, save_path + "label_encoder.pkl")
print(f"Model and Label Encoder successfully saved to Google Drive!")

Dataset loaded successfully!
Features: 39 | Classes: 29
Encoded class range: 0 to 28
Using device: cuda

Starting training on T4 GPU...
Epoch 01/15 | Loss: 0.4677 | Accuracy: 85.21%
Epoch 02/15 | Loss: 0.2356 | Accuracy: 88.00%
Epoch 03/15 | Loss: 0.2221 | Accuracy: 88.34%
Epoch 04/15 | Loss: 0.2159 | Accuracy: 88.31%
Epoch 05/15 | Loss: 0.2121 | Accuracy: 88.56%
Epoch 06/15 | Loss: 0.2107 | Accuracy: 88.44%
Epoch 07/15 | Loss: 0.2089 | Accuracy: 88.47%
Epoch 08/15 | Loss: 0.2081 | Accuracy: 88.59%
Epoch 09/15 | Loss: 0.2065 | Accuracy: 88.72%
Epoch 10/15 | Loss: 0.2058 | Accuracy: 88.73%
Epoch 11/15 | Loss: 0.2057 | Accuracy: 88.70%
Epoch 12/15 | Loss: 0.2038 | Accuracy: 88.72%
Epoch 13/15 | Loss: 0.2032 | Accuracy: 88.81%
Epoch 14/15 | Loss: 0.2034 | Accuracy: 88.82%
Epoch 15/15 | Loss: 0.2030 | Accuracy: 88.70%

--- Model Evaluation Report ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.96      1.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [2]:
from sklearn.utils.class_weight import compute_class_weight

# 1. Calculate class weights based on y_train distribution
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert weights to tensor and pass to GPU device
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# 2. Update Loss Function with weights
criterion = nn.CrossEntropyLoss(weight=weights_tensor)